# Module 4: Assembling Data with JOINs

**ALY 6420 | INNER, LEFT, RIGHT, FULL OUTER, Multi-Table, and Self-JOINs**

*Course Lecture Notes*


## Module 4

### Reconnecting data that belongs together

Relational databases deliberately separate information into related tables. A customer may be stored in one table, a rental event in another, and a film title in another. This design reduces redundancy, but analytical questions often require those pieces to be assembled again.

This module develops the reasoning needed to join tables correctly:

1. identify the tables that contain the needed facts
2. trace the key relationships between those tables
3. choose the JOIN type that matches the business question
4. predict which unmatched rows should remain or disappear
5. verify that one-to-many relationships have not inflated the result

> **Start here: trace the data path**
>
> Suppose a manager asks, “Which films has each customer rented?” Before writing SQL, identify where the customer name, rental event, inventory copy, and film title are stored. The query becomes much easier once the relationship path is clear.


## Learning objectives

By the end of this lecture, you should be able to:

- construct `INNER`, `LEFT`, `RIGHT`, and `FULL OUTER JOIN` queries
- explain which matched and unmatched rows each JOIN type returns
- write clear JOIN predicates using primary-key/foreign-key relationships
- use consistent table aliases and qualified column names
- trace and write multi-table JOIN chains
- explain why a missing JOIN condition creates a Cartesian product
- recognize an intentional `CROSS JOIN`
- use a self-JOIN when the same table must play two different roles
- identify one-to-many fan-out risk and check whether row counts or aggregations have been inflated
- diagnose common JOIN errors by reasoning from the data model and the expected result grain


## Required preparation

Use the following resources alongside this lecture.

### Principal resource

- Shan et al. (2025), *SQL for Data Analytics* (4th ed.), **Chapter 7: “Defining Datasets from Existing Datasets,” especially the section “Joining tables.”**
  - Focus on inner joins, outer joins, left/right/full outer joins, cross joins, and the chapter's business-oriented JOIN examples.
  - Chapter 7 also introduces derived datasets and set operations. Those topics provide useful context, but this module concentrates on JOINs.

### Course resources

- Module 4 Canvas content on JOIN types, multi-table JOINs, self-JOINs, and common JOIN pitfalls.
- PostgreSQL 16 Documentation, **7.2 Table Expressions**, for joined-table syntax and behavior.
- PostgreSQL Exercises, **Joins and Subqueries**, for additional practice.

### Companion database

All runnable examples in this lecture use the **Pagila** PostgreSQL database used throughout the course. Run the queries in DBeaver and inspect both the rows and the row counts.

> **Module 4 scope**
>
> The central question is not merely “Does the query run?” It is “Does this JOIN preserve and combine the rows required by the business question?”


# Part 1: Why JOINs Exist


## Normalization separates information on purpose

A relational database usually stores different entities in different tables.

In Pagila, examples include:

- `customer` — one row per customer
- `address` — one row per address
- `rental` — one row per rental event
- `inventory` — one row per physical inventory copy
- `film` — one row per film title

This separation is a consequence of relational modeling and normalization. It prevents the database from repeating the same customer, address, or film details in every transaction row.

The analytical consequence is important:

> **A well-designed database often requires JOINs precisely because related information is stored separately.**


## Foreign keys become JOIN paths

A foreign key records how one table relates to another.

For example:

```text
customer.address_id  →  address.address_id
rental.customer_id   →  customer.customer_id
rental.inventory_id  →  inventory.inventory_id
inventory.film_id    →  film.film_id
```

When you write a JOIN, the `ON` condition usually follows one of these relationships.

A useful way to think about it is:

```text
table-definition relationship  →  foreign key
query-time relationship        →  JOIN ... ON ...
```

The database design tells you which columns belong together.


## Start with the business question, not the syntax

Consider three questions:

1. **What is each customer's address?**
   - `customer` + `address`

2. **What films did a customer rent?**
   - `customer` + `rental` + `inventory` + `film`

3. **Which customers have never rented?**
   - `customer` + `rental`, but the query must preserve customers even when no rental exists

The third question needs a different JOIN type from the first two.

### Activity: identify the tables

For each question, write down:

- the requested output columns
- the table that owns each column
- the key path connecting the tables
- whether unmatched rows should remain in the result


## Result grain comes first

The **grain** of a result is what one row represents.

Examples:

- one row per customer
- one row per rental
- one row per customer-film rental
- one row per film title
- one row per actor pair per film

Before writing a JOIN, finish this sentence:

> **One row in my result should represent ________.**

This habit is one of the best defenses against duplicate-row surprises.


# Part 2: Anatomy of a JOIN


## Basic JOIN pattern

A two-table JOIN has four important pieces:

```sql
SELECT <columns>
FROM <left_table> AS l
INNER JOIN <right_table> AS r
    ON l.<key> = r.<key>;
```

Read it in this order:

1. begin with the left table
2. bring in the right table
3. compare rows using the `ON` predicate
4. return the requested columns

The JOIN type controls what happens when a match does **not** exist.


## The `ON` clause defines row matching

Example:

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    a.address
FROM customer AS c
INNER JOIN address AS a
    ON c.address_id = a.address_id;
```

The predicate

```sql
c.address_id = a.address_id
```

does not mean “compare any two similar-looking columns.”

It means “combine the customer row with the address row that represents that customer's address.”


## Use table aliases consistently

Long table names make multi-table queries difficult to read.

```sql
FROM customer AS c
INNER JOIN address AS a
    ON c.address_id = a.address_id
```

The aliases `c` and `a` let you write:

```sql
c.first_name
a.address
```

instead of repeating full table names.

Once a query involves multiple tables, qualified column references are especially useful because columns such as `customer_id`, `address_id`, and `film_id` may appear in more than one table.


## Why qualification prevents ambiguity

This is ambiguous:

```sql
SELECT address_id
FROM customer AS c
JOIN address AS a
    ON c.address_id = a.address_id;
```

Both tables contain `address_id`.

This is explicit:

```sql
SELECT
    c.address_id AS customer_address_id,
    a.address_id AS address_table_id
FROM customer AS c
JOIN address AS a
    ON c.address_id = a.address_id;
```

### Rule of thumb

In a multi-table query, qualify shared or potentially confusing column names with their table alias.


## Predict before you run

For this query:

```sql
SELECT
    c.customer_id,
    c.first_name,
    a.address
FROM customer AS c
INNER JOIN address AS a
    ON c.address_id = a.address_id;
```

Predict:

1. What does one result row represent?
2. Which table is the left table?
3. What must be true for a customer-address pair to appear?
4. If an address row has no customer, will that address appear?
5. Should the result contain more rows than the `customer` table? Why or why not?


# Part 3: INNER JOIN — Keep Only Matches


## INNER JOIN behavior

An `INNER JOIN` returns only row combinations that satisfy the JOIN predicate.

Conceptually:

```text
left table       right table
   A  ─────────────  A     keep
   B  ─────────────  B     keep
   C                 —     drop
   —                 D     drop
```

Rows without a match on the other side are excluded.

`JOIN` by itself is equivalent to `INNER JOIN` in PostgreSQL, but writing `INNER JOIN` explicitly can make your intent easier to read.


## INNER JOIN example: customer and address

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    a.address,
    a.postal_code
FROM customer AS c
INNER JOIN address AS a
    ON c.address_id = a.address_id
ORDER BY c.customer_id
LIMIT 20;
```

### Interpret the result

One row represents a customer paired with the address row referenced by that customer's `address_id`.

The JOIN keeps only rows where the two `address_id` values match.


## Real-world interpretation of INNER JOIN

Use an inner join when the analytical question is about records that exist on **both** sides of a relationship.

Examples:

- employees who have an assigned department
- orders that match an existing customer
- payments that correspond to a transaction
- students who have an enrollment record
- products that appear in an order line

The business question implicitly says:

> “Show me the cases where the relationship exists.”


## INNER JOIN can hide missing relationships

Suppose you are checking whether every active customer has a rental.

An inner join would be a poor first choice:

```sql
SELECT c.customer_id, r.rental_id
FROM customer AS c
INNER JOIN rental AS r
    ON c.customer_id = r.customer_id;
```

Why?

A customer with no rental simply disappears. The query cannot distinguish “customer does not exist” from “customer exists but has no matching rental” because both are absent from the result.

When missing matches are meaningful, use an outer join.


## Try it: INNER JOIN

Return each rental's:

- `rental_id`
- `rental_date`
- `customer_id`
- customer first name
- customer last name

Use `rental` and `customer`.

### Before running

Write the JOIN predicate from the foreign-key relationship.

. . .

```sql
SELECT
    r.rental_id,
    r.rental_date,
    c.customer_id,
    c.first_name,
    c.last_name
FROM rental AS r
INNER JOIN customer AS c
    ON r.customer_id = c.customer_id
ORDER BY r.rental_id
LIMIT 20;
```


# Part 4: LEFT JOIN — Preserve the Anchor Table


## LEFT JOIN behavior

A `LEFT JOIN` returns:

- every row from the left table
- matching rows from the right table
- `NULL` in right-table columns when no match exists

Conceptually:

```text
left table       right table
   A  ─────────────  A     keep match
   B  ─────────────  B     keep match
   C                 —     keep C + NULLs
   —                 D     do not keep D
```

The left table is often called the **anchor table** because it defines the population you want to preserve.


## LEFT JOIN example: customers and rentals

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    r.rental_id,
    r.rental_date
FROM customer AS c
LEFT JOIN rental AS r
    ON c.customer_id = r.customer_id
ORDER BY c.customer_id, r.rental_date;
```

The query preserves every customer.

If a customer has no matching rental, the customer columns remain populated and the rental columns are `NULL`.


## The anti-join pattern: find missing matches

A common analytical pattern is:

1. preserve all rows from the population of interest with `LEFT JOIN`
2. keep only rows where the right-side match is missing

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name
FROM customer AS c
LEFT JOIN rental AS r
    ON c.customer_id = r.customer_id
WHERE r.rental_id IS NULL;
```

This asks:

> Which customers have no matching rental row?


## Real-world anti-join questions

The same pattern can answer:

- customers with no orders
- employees with no submitted timesheet
- products with no sales
- students with no advising appointment
- claims with no payment record
- accounts with no recent login event

The key is to preserve the complete population on the left and then test for a missing right-side key.


## LEFT JOIN and aggregation: most recent rental

The Module 4 discussion asks for every active customer and the date of the customer's most recent rental, including customers who have never rented.

A suitable pattern is:

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    MAX(r.rental_date) AS last_rental
FROM customer AS c
LEFT JOIN rental AS r
    ON c.customer_id = r.customer_id
WHERE c.active = 1
GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name
ORDER BY c.customer_id;
```

`MAX()` summarizes the matching rental dates. If no rental matches, `MAX(r.rental_date)` remains `NULL`.

Aggregation is covered in greater depth later; here, focus on why the `LEFT JOIN` is necessary to preserve the customer population.


## A subtle pitfall: filtering the right table in `WHERE`

Consider:

```sql
SELECT
    c.customer_id,
    r.rental_date
FROM customer AS c
LEFT JOIN rental AS r
    ON c.customer_id = r.customer_id
WHERE r.rental_date >= DATE '2005-07-01';
```

The `LEFT JOIN` initially preserves customers with no rental, but those rows contain `NULL` in `r.rental_date`.

The `WHERE` condition then removes them.

So the query no longer preserves all customers.

### If preserving unmatched customers matters

Place the right-table condition inside the JOIN predicate:

```sql
LEFT JOIN rental AS r
    ON c.customer_id = r.customer_id
   AND r.rental_date >= DATE '2005-07-01'
```

Filter placement can change the meaning of an outer join.


# Part 5: RIGHT JOIN — The Mirror of LEFT JOIN


## RIGHT JOIN behavior

A `RIGHT JOIN` preserves every row from the right table.

```sql
SELECT
    c.first_name,
    c.last_name,
    a.address
FROM customer AS c
RIGHT JOIN address AS a
    ON c.address_id = a.address_id;
```

If an address has no matching customer, the address remains and the customer columns are `NULL`.


## RIGHT JOIN can usually be rewritten as LEFT JOIN

These two forms express the same preservation rule:

```sql
-- RIGHT JOIN
FROM customer AS c
RIGHT JOIN address AS a
    ON c.address_id = a.address_id
```

```sql
-- Equivalent LEFT JOIN
FROM address AS a
LEFT JOIN customer AS c
    ON c.address_id = a.address_id
```

The second version often reads more naturally because the table whose rows must all be preserved appears first.


## Why many analysts prefer LEFT JOIN

`LEFT JOIN` and `RIGHT JOIN` are both valid SQL.

A team may prefer `LEFT JOIN` because it keeps a consistent reading convention:

> “Start with the population I care about, then add related data.”

This is a style choice, not a difference in capability.

### Quick check

If a report must include **every address**, which table should be the left table if you want to use `LEFT JOIN`?


# Part 6: FULL OUTER JOIN — Preserve Both Sides


## FULL OUTER JOIN behavior

A `FULL OUTER JOIN` preserves:

- matched rows from both sides
- unmatched rows from the left
- unmatched rows from the right

```text
left only     → keep + NULLs on right
matched       → combine
right only    → keep + NULLs on left
```

This is useful when neither table should be treated as the only complete population.


## FULL OUTER JOIN syntax

```sql
SELECT
    c.customer_id,
    a.address_id,
    a.address
FROM customer AS c
FULL OUTER JOIN address AS a
    ON c.address_id = a.address_id;
```

In a database with enforced foreign keys, some kinds of unmatched rows may be impossible or rare. The JOIN type is still useful for reconciliation, staging tables, imported datasets, and data-quality checks where relationships may be incomplete.


## Real-world example: reconciling two systems

Suppose a health organization receives:

- a medical-visit dataset
- a dental-visit dataset

Some patients appear in both systems. Others appear in only one.

A `FULL OUTER JOIN` on patient identifier can preserve the complete set:

```sql
SELECT
    COALESCE(m.patient_id, d.patient_id) AS patient_id,
    m.last_medical_visit,
    d.last_dental_visit
FROM medical_visits AS m
FULL OUTER JOIN dental_visits AS d
    ON m.patient_id = d.patient_id;
```

This is a natural reconciliation use case because unmatched records on **either** side matter.


## Find unmatched rows on either side

A data-quality audit may focus only on mismatches:

```sql
SELECT
    c.customer_id,
    a.address_id
FROM customer AS c
FULL OUTER JOIN address AS a
    ON c.address_id = a.address_id
WHERE c.customer_id IS NULL
   OR a.address_id IS NULL;
```

Interpretation:

- `c.customer_id IS NULL` → right-side row without a left-side match
- `a.address_id IS NULL` → left-side row without a right-side match


# Part 7: CROSS JOIN and Cartesian Products


## CROSS JOIN matches every row with every row

A `CROSS JOIN` has no matching predicate.

```sql
SELECT
    f1.film_id AS film_1,
    f2.film_id AS film_2
FROM film AS f1
CROSS JOIN film AS f2;
```

If the first source has `m` rows and the second has `n` rows, the result has:

```text
m × n rows
```

This result is called a **Cartesian product**.


## A CROSS JOIN can be intentional

The textbook uses product-pair generation as an example of when every possible combination can be meaningful.

Related analytical uses include:

- generating product pairs for market-basket preparation
- creating every date-region combination for a reporting scaffold
- pairing scenarios with parameter values
- building test combinations

The key is that the all-combinations behavior must be **intentional**.


## Missing `ON` conditions create accidental Cartesian products

An accidental Cartesian product is one of the most dangerous JOIN mistakes.

Conceptually:

```sql
-- Do not use this as an ordinary relationship join
SELECT *
FROM customer AS c
CROSS JOIN address AS a;
```

If there are 599 customers and 603 addresses:

```text
599 × 603 = 361,197 rows
```

Most of those row combinations do not represent a real customer-address relationship.


## Diagnose a Cartesian product by row count

A common symptom is an unexpectedly enormous result.

Debugging routine:

1. count rows in the first table
2. count rows in the second table
3. multiply the counts
4. compare the product with the JOIN result count
5. inspect whether the intended `ON` predicate is missing or incorrect

### Quick check

If table A has 200 rows and table B has 500 rows, how many rows would an unrestricted Cartesian product contain?

. . .

**100,000 rows.**


## Try it safely with tiny inputs

You can observe Cartesian behavior without joining full large tables:

```sql
WITH a AS (
    SELECT *
    FROM film
    LIMIT 3
),
b AS (
    SELECT *
    FROM category
    LIMIT 2
)
SELECT
    a.title,
    b.name
FROM a
CROSS JOIN b;
```

Prediction:

```text
3 × 2 = 6 rows
```

The CTE syntax is used here only to keep the demonstration small; CTEs are studied in a later module.


# Part 8: Multi-Table JOINs


## Most business questions need more than two tables

A multi-table JOIN adds one relationship at a time.

General pattern:

```sql
SELECT ...
FROM table_a AS a
JOIN table_b AS b
    ON ...
JOIN table_c AS c
    ON ...
JOIN table_d AS d
    ON ...;
```

Each new table must connect to a table already in the JOIN chain.


## Trace the path before writing SQL

Pagila location path:

```text
customer
   │ address_id
   ▼
address
   │ city_id
   ▼
city
   │ country_id
   ▼
country
```

Write the relationships first:

```text
customer.address_id = address.address_id
address.city_id      = city.city_id
city.country_id      = country.country_id
```

Only then translate the path into SQL.


## Multi-table example: customer location

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    a.address,
    ci.city,
    co.country
FROM customer AS c
INNER JOIN address AS a
    ON c.address_id = a.address_id
INNER JOIN city AS ci
    ON a.city_id = ci.city_id
INNER JOIN country AS co
    ON ci.country_id = co.country_id
ORDER BY c.customer_id
LIMIT 30;
```

Read it as a sequence:

1. customer → address
2. address → city
3. city → country


## A second path: customer to film title

The rental path is:

```text
customer
   │ customer_id
   ▼
rental
   │ inventory_id
   ▼
inventory
   │ film_id
   ▼
film
```

There is no direct customer-to-film foreign key.

The path passes through the events and physical copies that connect them.


## Multi-table example: films rented by a customer

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    f.title,
    r.rental_date
FROM customer AS c
INNER JOIN rental AS r
    ON c.customer_id = r.customer_id
INNER JOIN inventory AS i
    ON r.inventory_id = i.inventory_id
INNER JOIN film AS f
    ON i.film_id = f.film_id
WHERE c.customer_id = 130
ORDER BY r.rental_date;
```

One row represents one rental event for the selected customer, enriched with the film title.


## Inventory acts as a bridge

A film title can have several inventory copies.

```text
film 1 ───< many inventory copies
inventory copy 1 ───< many rental events over time
```

The `inventory` table bridges film titles and rental events.

Without understanding this bridge, it is easy to write an impossible direct JOIN such as:

```sql
-- Wrong idea: rental does not carry film_id directly
ON rental.film_id = film.film_id
```

Always inspect the data model instead of guessing column paths.


## Multi-table JOIN checklist

Before running a long JOIN chain:

1. identify the result grain
2. list the required output columns
3. identify the table that owns each column
4. draw the key path
5. add one JOIN at a time
6. run and inspect after each step
7. compare row counts as the chain grows
8. verify that each `ON` predicate connects the intended keys


## Try it: actor, film, category

Return:

- actor first name
- actor last name
- film title
- category name

Trace the path:

```text
actor → film_actor → film → film_category → category
```

. . .

```sql
SELECT
    a.first_name,
    a.last_name,
    f.title,
    c.name AS category
FROM actor AS a
INNER JOIN film_actor AS fa
    ON a.actor_id = fa.actor_id
INNER JOIN film AS f
    ON fa.film_id = f.film_id
INNER JOIN film_category AS fc
    ON f.film_id = fc.film_id
INNER JOIN category AS c
    ON fc.category_id = c.category_id
ORDER BY a.last_name, a.first_name, f.title
LIMIT 50;
```


# Part 9: Self-JOINs


## A self-JOIN uses the same table twice

A self-JOIN is not a separate SQL keyword.

It is an ordinary JOIN in which the same table appears more than once with different aliases.

Conceptual example:

```sql
SELECT
    e.employee_name AS employee,
    m.employee_name AS manager
FROM employees AS e
LEFT JOIN employees AS m
    ON e.manager_id = m.employee_id;
```

The aliases are essential because the table plays two roles:

- `e` = employee
- `m` = manager


## Why aliases are mandatory in a self-JOIN

Without aliases, this is conceptually impossible to interpret:

```text
employees.employee_id
employees.employee_id
```

Which reference means “employee” and which means “manager”?

Aliases assign roles to each copy of the table.

The same idea applies when comparing:

- one transaction with another transaction
- one product with another product
- one actor-film row with another actor-film row


## Pagila self-JOIN: actors who share a film

`film_actor` maps actors to films.

Joining it to itself can create actor pairs that share the same film:

```sql
SELECT
    a1.first_name || ' ' || a1.last_name AS actor_1,
    a2.first_name || ' ' || a2.last_name AS actor_2,
    f.title AS film_title
FROM film_actor AS fa1
INNER JOIN film_actor AS fa2
    ON fa1.film_id = fa2.film_id
   AND fa1.actor_id < fa2.actor_id
INNER JOIN actor AS a1
    ON fa1.actor_id = a1.actor_id
INNER JOIN actor AS a2
    ON fa2.actor_id = a2.actor_id
INNER JOIN film AS f
    ON fa1.film_id = f.film_id
ORDER BY f.title
LIMIT 20;
```


## Why use `<` instead of `<>` in actor pairing?

If the condition were:

```sql
fa1.actor_id <> fa2.actor_id
```

both pair orders would appear:

```text
Actor A, Actor B
Actor B, Actor A
```

Using:

```sql
fa1.actor_id < fa2.actor_id
```

does two jobs:

- prevents an actor from pairing with themselves
- keeps only one ordering of each pair

This is a good example of a JOIN predicate doing more than simple key equality.


# Part 10: Cardinality, Grain, and Fan-Out


## A correct JOIN can still return more rows than expected

Not every large row count is a mistake.

If one customer has many rentals, joining:

```text
customer 1 ───< many rentals
```

naturally creates several result rows for that customer.

The relationship is **one-to-many**.

The important question is whether the repeated customer is appropriate for the intended result grain.


## Fan-out

**Fan-out** occurs when a row from one table matches multiple rows in another table, causing the first row's values to repeat.

Example:

```text
customer_id 10
    ├── rental 101
    ├── rental 102
    └── rental 103
```

After joining customer to rental, customer 10 appears three times.

That is correct if one row should represent a rental.

It is a problem if you expected one row per customer.


## Measure the relationship before trusting an aggregate

Suppose you join films to inventory:

```sql
SELECT
    f.film_id,
    f.title,
    i.inventory_id
FROM film AS f
JOIN inventory AS i
    ON f.film_id = i.film_id;
```

A film with five inventory copies appears five times.

If you later count rows, you are counting **film-copy combinations**, not distinct film titles.

Always ask:

> What does `COUNT(*)` count at the current grain?


## Compare total rows with distinct entities

A useful diagnostic:

```sql
SELECT
    COUNT(*) AS joined_rows,
    COUNT(DISTINCT f.film_id) AS distinct_films
FROM film AS f
JOIN inventory AS i
    ON f.film_id = i.film_id;
```

If `joined_rows` is much larger than `distinct_films`, that does not automatically mean the JOIN is wrong.

It tells you that the result grain is below the film level because films have multiple inventory rows.


## Fan-out becomes dangerous with multiple one-to-many joins

Imagine:

```text
customer
  ├── many rentals
  └── many payments
```

If you join both detail tables independently at the customer level, combinations can multiply.

For one customer:

```text
3 rentals × 4 payments = potentially 12 joined rows
```

If you then sum payment amounts without understanding the grain, totals may be inflated.

### Safe habit

Before aggregating after a JOIN:

- define the grain
- check relationship cardinality
- compare `COUNT(*)` with relevant `COUNT(DISTINCT ...)`
- pre-aggregate detail tables when the analytical grain requires it


## A row-count verification routine

After each JOIN step, record:

```sql
SELECT COUNT(*) ...
```

Then ask:

1. Did the row count stay the same?
2. Did it decrease because unmatched rows were dropped?
3. Did it increase because one row matched many?
4. Is that change consistent with the relationship?
5. Does the result still represent the intended grain?

A changing row count is evidence to interpret, not something to ignore.


# Part 11: Common JOIN Pitfalls


## Pitfall 1: missing or unintended JOIN condition

**Symptom:** result count is close to the product of the input table counts.

**Diagnosis:** Cartesian product.

**Fix:** write the relationship predicate explicitly.

```sql
SELECT *
FROM customer AS c
JOIN address AS a
    ON c.address_id = a.address_id;
```


## Pitfall 2: joining on the wrong column

A query may run successfully even when the relationship is wrong.

Example of a structurally valid but logically bad idea:

```sql
SELECT *
FROM customer AS c
JOIN rental AS r
    ON c.store_id = r.staff_id;
```

PostgreSQL can compare the values, but that does not make the columns relationally meaningful.

### Debugging question

What primary-key/foreign-key relationship is this JOIN supposed to follow?


## Pitfall 3: unexpected missing rows

**Symptom:** fewer rows than expected.

Possible causes:

- `INNER JOIN` dropped unmatched rows
- a later `WHERE` condition removed `NULL` outer-join rows
- the JOIN predicate is too restrictive
- the wrong table was used as the preserved side

### Debugging move

Run the left table by itself, then add the JOIN and compare row counts.


## Pitfall 4: unexpected duplicate-looking rows

**Symptom:** the same customer, film, or product appears repeatedly.

Possible causes:

- the relationship is legitimately one-to-many
- you selected only high-level columns and hid the detail column that distinguishes rows
- multiple one-to-many joins created fan-out

### Debugging move

Add the keys from the detail tables to the `SELECT` list so you can see what makes the rows distinct.


## Pitfall 5: ambiguous column references

**Symptom:** PostgreSQL reports that a column reference is ambiguous.

```sql
SELECT customer_id
FROM customer AS c
JOIN rental AS r
    ON c.customer_id = r.customer_id;
```

Both tables contain `customer_id`.

Fix:

```sql
SELECT c.customer_id
...
```

Qualification is part of readable JOIN design, not just error repair.


## Pitfall 6: `SELECT *` hides the real structure

`SELECT *` is useful while exploring a relationship, but wide joined outputs can become difficult to interpret.

A better final query usually names the columns needed for the question:

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    r.rental_date
FROM customer AS c
JOIN rental AS r
    ON c.customer_id = r.customer_id;
```

Selecting intentionally also makes duplicate column names easier to manage.


## JOIN debugging checklist

When a JOIN result surprises you, check in this order:

1. **Business question** — What population must be preserved?
2. **Grain** — What should one row represent?
3. **Tables** — Does each requested field come from the right source?
4. **Path** — Are you following real key relationships?
5. **JOIN type** — Should unmatched rows survive?
6. **Predicate** — Are the correct columns compared?
7. **Filters** — Did `WHERE` remove outer-join rows?
8. **Cardinality** — Is one-to-many fan-out expected?
9. **Counts** — How did the row count change at each step?


# Part 12: Combining JOINs with Earlier SELECT Skills


## JOINs define the source; WHERE filters the joined result

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    r.rental_date
FROM customer AS c
INNER JOIN rental AS r
    ON c.customer_id = r.customer_id
WHERE r.rental_date >= TIMESTAMP '2005-07-01 00:00:00'
ORDER BY r.rental_date DESC
LIMIT 25;
```

Read the logic:

1. combine matching customers and rentals
2. keep rows meeting the date condition
3. choose output columns
4. sort
5. limit displayed rows


## Filter the correct table

Suppose the question is:

> Which active customers rented films after July 1, 2005?

Relevant conditions belong to different tables:

```sql
WHERE c.active = 1
  AND r.rental_date >= TIMESTAMP '2005-07-01 00:00:00'
```

Always qualify filter columns in multi-table queries when doing so improves clarity.


## Build complex queries incrementally

Instead of writing five JOINs at once:

```text
Step 1: run the anchor table
Step 2: add one JOIN and inspect
Step 3: add the next JOIN and inspect
Step 4: select the required columns
Step 5: add filters
Step 6: add sorting
Step 7: verify row count and grain
```

This makes it much easier to locate the step where a result changes unexpectedly.


# Part 13: Guided Practice


## Practice 1: customer and address

Write a query that returns:

- customer ID
- first name
- last name
- address
- postal code

Use an inner join.

### Reflection

What key connects the two tables?


## Practice 2: all customers and any rentals

Return every customer with any rental records they have.

Include:

- customer ID
- first name
- last name
- rental ID
- rental date

### Requirement

Customers with no rental must remain.

Which JOIN type expresses that requirement?


## Practice 3: customers with no rentals

Modify Practice 2 so the output contains only customers with no matching rental.

### Hint

Test a right-side key for `NULL`.


## Practice 4: rewrite RIGHT as LEFT

Start with:

```sql
SELECT
    c.first_name,
    c.last_name,
    a.address
FROM customer AS c
RIGHT JOIN address AS a
    ON c.address_id = a.address_id;
```

Rewrite it with `LEFT JOIN` while preserving the same rows.

### Reflection

What changes? What stays the same?


## Practice 5: customer to country

Return each customer's:

- full name
- address
- city
- country

Do not begin by typing SQL.

First draw the table path and write the three key relationships.


## Practice 6: rental to film

Return:

- rental ID
- rental date
- film title

Trace:

```text
rental → inventory → film
```

### Verification

What does one output row represent?


## Practice 7: diagnose fan-out

Run:

```sql
SELECT
    f.film_id,
    f.title,
    i.inventory_id
FROM film AS f
JOIN inventory AS i
    ON f.film_id = i.film_id
ORDER BY f.film_id;
```

Then answer:

1. Why can a film appear multiple times?
2. Which column reveals the lower-grain records?
3. Would `COUNT(*)` equal the number of film titles?
4. What would `COUNT(DISTINCT f.film_id)` measure?


## Practice 8: intentional combinations

Create a small set of three films and two categories, then use a cross join.

Before running, predict the row count.

### Reflection

How is this different from accidentally forgetting a relationship condition in a normal JOIN?


# Part 14: Debugging and AI Critique


## A query can be syntactically valid and analytically wrong

Consider an AI-generated query for:

> Return each customer's address.

```sql
SELECT
    c.customer_id,
    c.first_name,
    a.address
FROM customer AS c
CROSS JOIN address AS a;
```

The SQL is valid.

The analysis is wrong because every customer is paired with every address.

### Critique

Explain the error in terms of:

- the data model
- expected grain
- row count
- missing relationship predicate


## Correct the Cartesian-product query

A correct relationship JOIN is:

```sql
SELECT
    c.customer_id,
    c.first_name,
    a.address
FROM customer AS c
INNER JOIN address AS a
    ON c.address_id = a.address_id;
```

The important correction is not simply adding `ON`.

It is adding the **correct relational condition**.


## Second AI critique: wrong JOIN type

Question:

> Show every active customer and the date of their most recent rental, including customers who have never rented.

AI response:

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    MAX(r.rental_date) AS last_rental
FROM customer AS c
INNER JOIN rental AS r
    ON c.customer_id = r.customer_id
WHERE c.active = 1
GROUP BY c.customer_id, c.first_name, c.last_name;
```

### What is wrong?

The aggregation is plausible, but the `INNER JOIN` removes customers with no rental before `MAX()` is calculated.

The correct preservation rule requires a `LEFT JOIN`.


## Third AI critique: hidden fan-out

Suppose an AI joins `customer`, `rental`, and `payment` and then sums payment amounts, but it never states the result grain or checks relationship cardinality.

Questions to ask:

1. Which key connects each table?
2. Can one customer have many rentals?
3. Can one customer have many payments?
4. Are detail rows being multiplied?
5. What does each joined row represent?
6. Should any table be aggregated before joining?

A query that runs is only the beginning of validation.


# Part 15: Lab 3 Readiness


## What Lab 3 expects you to demonstrate

Lab 3 focuses on assembling data with JOINs.

You should be ready to:

- choose an appropriate JOIN type from a business requirement
- write `INNER`, `LEFT`, and `FULL OUTER JOIN` queries
- trace foreign-key paths across multiple tables
- use aliases consistently
- interpret `NULL` values produced by outer joins
- distinguish intentional from accidental Cartesian products
- critique a plausible but incorrect JOIN


## Lab habit: translate the requirement before coding

For every JOIN task, write four lines before SQL:

```text
Population to preserve:
Output grain:
Tables needed:
Key path:
```

Then decide the JOIN type.

Example:

```text
Population to preserve: every customer
Output grain: one row per customer-rental match, or unmatched customer
Tables needed: customer, rental
Key path: customer.customer_id = rental.customer_id
JOIN type: LEFT JOIN
```


## Lab habit: verify the result

Do not stop after successful execution.

Check:

- expected columns
- row count
- presence or absence of `NULL`
- duplicate-looking entities
- whether unmatched rows were preserved correctly
- whether a one-to-many relationship changed the grain
- whether sample rows make business sense


# Part 16: Discussion Preparation


## Choosing the right JOIN

Discussion scenario:

> A manager wants every active customer and the date of the customer's most recent rental. Customers who have never rented must still appear.

The key phrase is:

> **every active customer**

That defines the population to preserve.

Therefore:

```text
left table  = customer
right table = rental
JOIN type   = LEFT JOIN
```

An `INNER JOIN` would remove active customers with no rental.


## Why a RIGHT JOIN can be equivalent

These structures can preserve the same customer population:

```sql
FROM customer AS c
LEFT JOIN rental AS r
    ON c.customer_id = r.customer_id
```

```sql
FROM rental AS r
RIGHT JOIN customer AS c
    ON c.customer_id = r.customer_id
```

The preserved table is `customer` in both cases.

The difference is only which side of the syntax the table occupies.

A strong explanation names the preservation rule rather than saying only that “LEFT and RIGHT are opposites.”


# Part 17: Check Your Understanding


## Knowledge check 1

A query must return only customers who have made at least one rental.

Which JOIN type is the natural default?

A. `LEFT JOIN`  
B. `INNER JOIN`  
C. `FULL OUTER JOIN`  
D. `CROSS JOIN`

. . .

**Answer: B. `INNER JOIN`**

The question requires matching customer-rental relationships only.


## Knowledge check 2

A query must return every customer, including customers with no rental.

Which structure best expresses the requirement?

A. `customer INNER JOIN rental`  
B. `customer LEFT JOIN rental`  
C. `customer CROSS JOIN rental`  
D. `rental LEFT JOIN customer`

. . .

**Answer: B. `customer LEFT JOIN rental`**


## Knowledge check 3

What does this condition do?

```sql
WHERE r.rental_id IS NULL
```

when used after:

```sql
FROM customer AS c
LEFT JOIN rental AS r
    ON c.customer_id = r.customer_id
```

. . .

It keeps customers for whom no matching rental row was found.


## Knowledge check 4

Why can this query return more rows than the `film` table?

```sql
SELECT f.film_id, f.title, i.inventory_id
FROM film AS f
JOIN inventory AS i
    ON f.film_id = i.film_id;
```

. . .

Because one film can have multiple inventory copies. The result grain is one film-inventory combination, not one film.


## Knowledge check 5

What is the main risk of a missing JOIN predicate?

. . .

A Cartesian product: every row from one input can pair with every row from the other, creating a very large and usually meaningless result.


## Knowledge check 6

Why does this path require `inventory`?

```text
customer → rental → inventory → film
```

. . .

`rental` stores `inventory_id`, not the film title or direct `film_id`. `inventory` connects the rented physical copy to the corresponding film.


## Knowledge check 7

What is the safest first question when duplicate-looking rows appear after a JOIN?

A. “How do I remove duplicates?”  
B. “Should I add `DISTINCT`?”  
C. “What does one row represent at the current grain?”  
D. “Should I switch every JOIN to LEFT JOIN?”

. . .

**Answer: C.**

Repeated values may be legitimate consequences of a one-to-many relationship. Understand the grain before removing anything.


## Knowledge check 8

A `RIGHT JOIN` preserves all rows from which table?

. . .

The right table — the table named after `RIGHT JOIN`.

An equivalent result can usually be written as a `LEFT JOIN` by swapping table order.


# Part 18: Wrap-up


## Module 4 concept map

```text
Relational model
      │
      ├── primary keys
      └── foreign keys
              │
              ▼
        JOIN predicates
              │
      ┌───────┼────────┬─────────────┐
      ▼       ▼        ▼             ▼
   INNER     LEFT     RIGHT        FULL OUTER
 matches   keep L    keep R         keep both
      │       │        │             │
      └───────┴────────┴─────────────┘
              │
              ▼
       multi-table paths
              │
              ├── aliases
              ├── bridge tables
              └── self-JOIN roles
              │
              ▼
        verify result grain
              │
              ├── row counts
              ├── NULL behavior
              ├── cardinality
              └── fan-out
```


## Module 4 takeaways

- JOINs reconstruct relationships that normalized relational design stores across separate tables.
- The `ON` predicate should follow a meaningful relationship, usually a primary-key/foreign-key path.
- `INNER JOIN` keeps matches only.
- `LEFT JOIN` preserves the left-side population.
- `RIGHT JOIN` is the mirror of `LEFT JOIN` and can usually be rewritten by swapping table order.
- `FULL OUTER JOIN` preserves unmatched rows from both sides and is useful for reconciliation.
- `CROSS JOIN` creates every possible row combination and should be intentional.
- Multi-table JOINs are easiest to write by tracing the foreign-key path before writing SQL.
- Self-JOINs use aliases so one table can play multiple roles.
- One-to-many relationships can increase row counts; always verify grain and fan-out before trusting aggregates.


## Before you move on

You should be able to explain, without running SQL:

1. which table population a JOIN must preserve
2. why a particular JOIN type matches the business requirement
3. what the `ON` condition means relationally
4. what one result row represents
5. why the row count increased, decreased, or stayed the same
6. whether `NULL` means “no match” in the current outer join
7. whether repeated entities are expected one-to-many behavior or evidence of a mistake
8. how to trace a multi-table path from one requested field to another

If any of these are unclear, revisit the corresponding part before Lab 3.


## References and course resources

### Principal text

Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for data analytics: Analyze data effectively, uncover insights and master advanced SQL for real-world applications* (4th ed.). Packt Publishing. Chapter 7, “Defining Datasets from Existing Datasets.”

### Course and supporting resources

- ALY 6420 Module 4 Canvas content: JOIN types, multi-table JOINs, self-JOINs, and common pitfalls.
- PostgreSQL Global Development Group. PostgreSQL 16 Documentation, Section 7.2, “Table Expressions.”
- PostgreSQL Global Development Group. PostgreSQL 16 Documentation, Section 14.3, “Controlling the Planner with Explicit JOIN Clauses.”
- PostgreSQL Exercises. “Joins and Subqueries.”

### Database used in examples

- Pagila sample database, queried through PostgreSQL in DBeaver.
